In [0]:
from azure.storage.blob import BlobServiceClient

In [0]:
%run ../Configurations/Init_Scripts

0
0
0


jdbc:sqlserver://usazrsqld1027.database.windows.net:1433;database=CS-DataMonitoring;queryTimeout=30


In [0]:
# source connection string
agn_storage_connection_string = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="AGNUSBFileStore")
# destination connection string
abbvie_storage_connection_string = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="DefenderBlobStorageConnection")
source_container_name = "usbupload-logs"
destination_container_name = "usbuploaded-zip"
# Log record
defaultValue = "0"

dbutils.widgets.text('job_id',defaultValue)
job_id = dbutils.widgets.get('job_id')

dbutils.widgets.text('run_id',defaultValue)
run_id = dbutils.widgets.get('run_id')

dbutils.widgets.text("configId", "36")
configId = dbutils.widgets.get("configId")

dbutils.widgets.text("sourceTypeId", "11")
sourceTypeId = dbutils.widgets.get("sourceTypeId")

dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")
 
dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text('ExternalLocation_silver',"/mntprod_silver")
ExternalLocation_silver = dbutils.widgets.get('ExternalLocation_silver')


try:
    # Source BlobServiceClient    
    source_blob_service_client = BlobServiceClient.from_connection_string(agn_storage_connection_string)
    source_container_client = source_blob_service_client.get_container_client(source_container_name)

    # Destination BlobServiceClient
    destination_blob_service_client = BlobServiceClient.from_connection_string(abbvie_storage_connection_string)
    destination_container_client = destination_blob_service_client.get_container_client(destination_container_name)

    # List blobs in the source container
    blobs = source_container_client.list_blobs()
    # Move each blob from source to destination
    for blob in blobs:      
        source_blob_client = source_blob_service_client.get_blob_client(container=source_container_name, blob=blob.name)
        content = source_blob_client.download_blob().readall()

        destination_blob_client = destination_blob_service_client.get_blob_client(container=destination_container_name, blob=blob.name)
        destination_blob_client.upload_blob(content, overwrite=True)   

        # Delete blob from source
        source_blob_client.delete_blob()
        
        processedFileList =([{'ConfigId':configId,'SourceTypeId':sourceTypeId,'SourceContainerPath':source_container_name,
                            'SourceFileName':blob.name,'DestinationContainerPath':destination_container_name,'DestinationFileName':blob.name, 'LoadType':"USB", 'PipelineStatus':"Succeeded",'PipelineRunId':run_id, 'JobId':job_id
                            }])      
        logIntoIngestionLogTable(processedFileList,'MoveFilesfromAllergantoAbbvie')
except Exception as exp:    
    processedFileList =([{'ConfigId':configId,'SourceTypeId':sourceTypeId,'SourceContainerPath':source_container_name,
                            'DestinationContainerPath':destination_container_name, 'DestinationFileName':blob.name,'LoadType':"USB",'PipelineStatus':"Failed",'PipelineRunId':run_id, 'JobId':job_id, 'ErrorMessage': str(exp)
                            }])
    logIntoIngestionLogTable(processedFileList,'MoveFilesfromAllergantoAbbvie')


    